# 📊 Proyecto Final - Sistema Inteligente de Recomendación para E-commerce

# Notebook 06: Sistema de Recomendación

---

## 👥 Equipo

- Yustin Rodríguez
- Elías
- Carina
- Rocío

---

## 🎯 Objetivo

Diseñar, entrenar e implementar un sistema de recomendación capaz de sugerir productos personalizados a los usuarios a partir de su historial de navegación, compras, preferencias e interacciones registradas en la plataforma de comercio electrónico.

Este modelo constituye la solución principal del proyecto y busca mejorar la experiencia del usuario mediante recomendaciones relevantes y personalizadas.

---

## 📌 Objetivos específicos

- Construir modelos de recomendación basados en diferentes enfoques.
- Comparar el desempeño de cada modelo.
- Seleccionar el modelo con mejores resultados.
- Generar recomendaciones personalizadas para cada usuario.

---

## 📂 Datos utilizados

- Dataset preparado en el Notebook 04.
- Variables creadas durante Feature Engineering.
- Información de clientes.
- Productos.
- Órdenes.
- Eventos.
- Reviews.

---

## 📋 Modelos implementados

| Integrante | Modelo |
|------------|---------|
| **Yustin** | Baseline (Popularidad) |
| **Elías** | Content-Based Filtering |
| **Carina** | Collaborative Filtering |
| **Rocío** | Modelo híbrido y comparación final |

---

## 📈 Resultado esperado

Desarrollar un sistema capaz de generar recomendaciones relevantes para cada usuario utilizando diferentes técnicas de recomendación y seleccionar el modelo con mejor desempeño para su implementación.

---

**Metodología:** CRISP-DM

**Sprint:** 1 – Modelado

## 🔧 Flujo de entrenamiento con LightGBM y XGBoost

Este notebook implementa un primer pipeline de recomendación usando datos preparados previamente por el feature engineering.

### Pasos que se van a realizar
1. Cargar la matriz usuario-producto y los features derivados.
2. Construir ejemplos positivos y negativos para entrenar un modelo de clasificación binaria.
3. Separar los datos en train y test.
4. Entrenar dos modelos:
   - LightGBM
   - XGBoost
5. Medir el desempeño con:
   - accuracy
   - precision
   - recall
   - f1
   - MAP@K
   - NDCG@K
6. Comparar los resultados para elegir el modelo inicial más prometedor.

La idea es empezar con un enfoque simple y reproducible antes de pasar a modelos más complejos.

In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

sys.path.append(str(NOTEBOOK_DIR.parent))

from SRC.feature_engineering import generar_features

interaction_matrix, product_features, user_features, user_item_df, events_con_user = generar_features()

print("Matriz usuario-item:", interaction_matrix.shape)
print("Features producto:", product_features.shape)
print("Features usuario:", user_features.shape)
print("DataFrame usuario-item:", user_item_df.shape)


############################################################
# PIPELINE DE FEATURE ENGINEERING
############################################################


PASO 1: Conversión de columnas de fechas a datetime
  ✓ orders['order_time'] → datetime64[ns]
  ✓ customers['signup_date'] → datetime64[ns]

--- Limpiando customers ---
  ✓ Sin valores nulos: OK (20,000 registros)
  ✓ Sin filas duplicadas: OK (20,000 registros)
  ✓ age entre 18 y 100: OK (20,000 registros)
  ✓ email con formato válido: OK (20,000 registros)

--- Limpiando products ---
  ✓ Sin valores nulos: OK (1,197 registros)
  ✓ Sin filas duplicadas: OK (1,197 registros)
  ✓ price_usd > 0: OK (1,197 registros)
  ✓ cost_usd > 0: OK (1,197 registros)
  ✓ cost_usd <= price_usd: OK (1,197 registros)
  ✓ margin_usd coincide con price_usd - cost_usd (tolerancia 0.011): OK (1,197 registros)

--- Corrigiendo coherencia temporal (orders vs customers) ---
  order_time < signup_date: 16,923/33,580 registros
  → Corregidas 16,923 fechas de order_time (signup_date + offset aleatorio 0-365 días)

--- Limpiando orders ---
  ✓ Sin valores nulos: OK (33,580 registros)
  ✓ Sin filas duplicadas: OK (3

  Usuarios: 19,945
  Productos: 1,197
  Interacciones: 529,593
  Esparsidad: 97.78%

PASO 2: Features de producto


  Productos: 1,197
  Features: ['category', 'price_usd', 'cost_usd', 'margin_usd', 'n_views', 'n_cart', 'n_purchases', 'popularidad', 'rating_promedio', 'n_ratings']

PASO 3: Features de usuario


  Usuarios: 20,000
  Features: ['age', 'country', 'marketing_opt_in', 'n_sessions', 'n_purchases', 'ticket_promedio', 'n_products_viewed', 'n_products_carted', 'rating_promedio_usr']

PASO 4: Preprocesamiento para modelado


  Matriz dispersa: 19945 usuarios x 1197 productos
  Features producto: 12 columnas
  Features usuario: 11 columnas
  DataFrame usuario-item: 529,593 filas

############################################################
# FEATURE ENGINEERING COMPLETADO
############################################################

  Matriz usuario-item: (19945, 1197)
  Features producto:   (1197, 12)
  Features usuario:    (20000, 11)



Archivos guardados en: C:\Users\Usuario\Documents\Henry\TPfinal\tp final\ecommerce-clickstream-ml\Data\Processed
Matriz usuario-item: (19945, 1197)
Features producto: (1197, 12)
Features usuario: (20000, 11)
DataFrame usuario-item: (529593, 3)


## Entrenamiento inicial: LightGBM vs XGBoost

En esta etapa se construyen ejemplos positivos y negativos a partir de las interacciones observadas, se agregan los features de usuario y producto y se entrenan dos modelos de clasificación binaria para predecir si un usuario interactuará con un producto.

El split entre train y test es **temporal** (no aleatorio): se ordenan las interacciones por fecha y las últimas ~20% quedan como test, simulando predecir comportamiento futuro. Los features de usuario/producto se recalculan usando solo información previa al corte, para evitar que el modelo "vea" datos del período que tiene que predecir. Los negativos se muestrean con probabilidad proporcional a la popularidad del producto (negativos "duros"), y se usa un peso balanceado por clase al entrenar, porque hay muchos más positivos que negativos.

El objetivo es obtener una primera comparación rápida y reproducible antes de avanzar a modelos más complejos.

In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from ngboost import NGBoost
from ngboost.distns import Bernoulli

from SRC.feature_engineering import crear_features_producto, crear_features_usuario

DATA_DIR = Path.cwd().parent / 'Data' / 'Processed'
RAW_DIR = Path.cwd().parent / 'Data' / 'Raw'

user_item_df = pd.read_csv(DATA_DIR / 'user_item_df.csv')

products_raw = pd.read_csv(RAW_DIR / 'products.csv')
customers_raw = pd.read_csv(RAW_DIR / 'customers.csv')
sessions_raw = pd.read_csv(RAW_DIR / 'sessions.csv', parse_dates=['start_time'])
orders_raw = pd.read_csv(RAW_DIR / 'orders.csv', parse_dates=['order_time'])
order_items_raw = pd.read_csv(RAW_DIR / 'order_items.csv')
reviews_raw = pd.read_csv(RAW_DIR / 'reviews.csv', parse_dates=['review_time'])

events_con_user['timestamp'] = pd.to_datetime(events_con_user['timestamp'])

print('user_item_df shape:', user_item_df.shape)
print('products_raw shape:', products_raw.shape)
print('customers_raw shape:', customers_raw.shape)


user_item_df shape: (529593, 3)
products_raw shape: (1197, 6)
customers_raw shape: (20000, 7)


In [3]:
# Split temporal: usamos el timestamp de la última interacción de cada par
# usuario-producto para separar interacciones "pasadas" (train) de "futuras" (test).
# Esto simula el escenario real (predecir comportamiento que todavía no pasó) en
# vez de mezclar pares al azar, que podían compartir usuario y producto entre train y test.

positive_pairs = user_item_df[user_item_df['score'] > 0][['customer_id', 'product_id']].copy()
positive_pairs['label'] = 1

interaction_times = (
    events_con_user
    .dropna(subset=['customer_id', 'product_id'])
    .assign(
        customer_id=lambda d: d['customer_id'].astype(int),
        product_id=lambda d: d['product_id'].astype(int),
    )
    .groupby(['customer_id', 'product_id'])['timestamp']
    .max()
    .reset_index()
)

positive_pairs = positive_pairs.merge(interaction_times, on=['customer_id', 'product_id'], how='left')
positive_pairs = positive_pairs.dropna(subset=['timestamp'])

cutoff = positive_pairs['timestamp'].quantile(0.8)
print('Corte temporal (80% train / 20% test):', cutoff)

train_positive = positive_pairs[positive_pairs['timestamp'] <= cutoff].drop(columns='timestamp')
test_positive = positive_pairs[positive_pairs['timestamp'] > cutoff].drop(columns='timestamp')

print('Positivos train:', len(train_positive), '| Positivos test:', len(test_positive))

# Features de usuario/producto recalculados usando SOLO datos anteriores al corte,
# para que ningún feature "vea" información del período de test (evita leakage).
events_train = events_con_user[events_con_user['timestamp'] <= cutoff]
sessions_train = sessions_raw[sessions_raw['start_time'] <= cutoff]
orders_train = orders_raw[orders_raw['order_time'] <= cutoff]
order_items_train = order_items_raw[order_items_raw['order_id'].isin(orders_train['order_id'])]
reviews_train = reviews_raw[reviews_raw['review_time'] <= cutoff]

product_features_train = crear_features_producto(
    products_raw, events_train, reviews_train, order_items_train, orders_train
)
user_features_train = crear_features_usuario(
    customers_raw, sessions_train, events_train, orders_train, order_items_train, reviews_train
)

Corte temporal (80% train / 20% test): 2024-09-07 23:12:08.600000
Positivos train: 423674 | Positivos test: 105919

PASO 2: Features de producto


  Productos: 1,197
  Features: ['category', 'price_usd', 'cost_usd', 'margin_usd', 'n_views', 'n_cart', 'n_purchases', 'popularidad', 'rating_promedio', 'n_ratings']

PASO 3: Features de usuario


  Usuarios: 20,000
  Features: ['age', 'country', 'marketing_opt_in', 'n_sessions', 'n_purchases', 'ticket_promedio', 'n_products_viewed', 'n_products_carted', 'rating_promedio_usr']


In [4]:
# Negativos "duros": pares usuario-producto no observados, muestreados con
# probabilidad proporcional a la popularidad del producto en el período de train
# (en vez de uniforme al azar). Así el modelo no puede resolver la tarea solo con
# "¿este producto es popular?" -- tiene que aprender afinidad usuario-producto real.

rng = np.random.default_rng(42)

customer_ids_unique = customers_raw['customer_id'].to_numpy()
product_ids_unique = products_raw['product_id'].to_numpy()

popularidad_train = product_features_train.set_index('product_id')['popularidad']
pesos_producto = popularidad_train.reindex(product_ids_unique).fillna(0).to_numpy() + 1.0
probs_producto = pesos_producto / pesos_producto.sum()

observed_pairs = set(map(tuple, positive_pairs[['customer_id', 'product_id']].to_numpy()))


def samplear_negativos_duros(n, customer_pool, product_pool, product_probs, observed, rng):
    negativos = set()
    while len(negativos) < n:
        faltan = n - len(negativos)
        candidatos_customer = rng.choice(customer_pool, size=faltan * 2)
        candidatos_product = rng.choice(product_pool, size=faltan * 2, p=product_probs)
        for cid, pid in zip(candidatos_customer, candidatos_product):
            par = (int(cid), int(pid))
            if par not in observed and par not in negativos:
                negativos.add(par)
                if len(negativos) >= n:
                    break
    return list(negativos)


n_neg_train = min(len(train_positive), 100_000)
n_neg_test = min(len(test_positive), 30_000)

neg_train = samplear_negativos_duros(
    n_neg_train, customer_ids_unique, product_ids_unique, probs_producto, observed_pairs, rng
)
observed_pairs.update(neg_train)
neg_test = samplear_negativos_duros(
    n_neg_test, customer_ids_unique, product_ids_unique, probs_producto, observed_pairs, rng
)

train_negative = pd.DataFrame(neg_train, columns=['customer_id', 'product_id'])
train_negative['label'] = 0
test_negative = pd.DataFrame(neg_test, columns=['customer_id', 'product_id'])
test_negative['label'] = 0

train_pairs = pd.concat([train_positive, train_negative], axis=0, ignore_index=True)
test_pairs = pd.concat([test_positive, test_negative], axis=0, ignore_index=True)

print('Train (label -> conteo):', train_pairs['label'].value_counts().to_dict())
print('Test (label -> conteo):', test_pairs['label'].value_counts().to_dict())

# Unión de features (calculados solo con datos de train) a los pares de train y test
user_features_model = user_features_train.rename(columns={'n_purchases': 'n_purchases_user'})
product_features_model = product_features_train.rename(columns={'n_purchases': 'n_purchases_product'})

feature_cols = [
    'age', 'country', 'marketing_opt_in', 'n_sessions', 'n_purchases_user', 'ticket_promedio',
    'n_products_viewed', 'n_products_carted', 'rating_promedio_usr',
    'price_usd', 'cost_usd', 'margin_usd', 'popularidad', 'rating_promedio', 'n_views',
    'n_cart', 'n_purchases_product', 'n_ratings', 'category'
]


def construir_features(pares_df):
    df = pares_df.merge(user_features_model, on='customer_id', how='left')
    df = df.merge(product_features_model, on='product_id', how='left')
    return df


train_df = construir_features(train_pairs)
test_df = construir_features(test_pairs)

feature_cols = [col for col in feature_cols if col in train_df.columns]

# Codificación de categóricas: se ajusta solo con train y se aplica a test
# (una categoría no vista en train se mapea a una clase "desconocida" aparte)
for col in ['country', 'category']:
    train_df[col] = train_df[col].fillna('missing').astype(str)
    test_df[col] = test_df[col].fillna('missing').astype(str)

    categorias = sorted(train_df[col].unique())
    mapping = {cat: i for i, cat in enumerate(categorias)}
    desconocida = len(categorias)

    train_df[col] = train_df[col].map(mapping)
    test_df[col] = test_df[col].map(mapping).fillna(desconocida).astype(int)

X_train = train_df[feature_cols].copy()
y_train = train_df['label']
X_test = test_df[feature_cols].copy()
y_test = test_df['label']

# Imputación ajustada solo con train
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=feature_cols, index=X_test.index)

# Pesos balanceados: compensan que hay muchos más positivos que negativos
sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

print('Tamaño train:', X_train.shape)
print('Tamaño test:', X_test.shape)
print('Distribución y_train:', y_train.value_counts().to_dict())
print('Distribución y_test:', y_test.value_counts().to_dict())


Train (label -> conteo): {1: 423674, 0: 100000}
Test (label -> conteo): {1: 105919, 0: 30000}


Tamaño train: (523674, 19)
Tamaño test: (135919, 19)
Distribución y_train: {1: 423674, 0: 100000}
Distribución y_test: {1: 105919, 0: 30000}


In [5]:
def evaluar_modelo(modelo, nombre, sample_weight=None):
    modelo.fit(X_train, y_train, sample_weight=sample_weight)
    pred = modelo.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }

    print(f'\n=== {nombre} ===')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}')

    return metrics

lightgbm_model = lgb.LGBMClassifier(
    n_estimators=20,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

xgboost_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

catboost_model = CatBoostClassifier(
    iterations=50,
    learning_rate=0.1,
    depth=3,
    verbose=False,
    random_state=42,
)

ngboost_model = NGBoost(
    Dist=Bernoulli,
    n_estimators=50,
    learning_rate=0.1,
    natural_gradient=True,
    verbose=False,
    random_state=42,
)

models_to_test = [
    ('LightGBM', lightgbm_model),
    ('XGBoost', xgboost_model),
    ('CatBoost', catboost_model),
    ('NGBoost', ngboost_model),
]

for nombre, modelo in models_to_test:
    evaluar_modelo(modelo, nombre, sample_weight=sample_weight_train)


[LightGBM] [Info] Number of positive: 423674, number of negative: 100000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.097454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2004
[LightGBM] [Info] Number of data points in the train set: 523674, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf



=== LightGBM ===
accuracy: 0.4970
precision: 0.8017
recall: 0.4711
f1: 0.5935


C:\Users\Usuario\Documents\Henry\TPfinal\tp final\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:11:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== XGBoost ===
accuracy: 0.5011
precision: 0.8047
recall: 0.4751
f1: 0.5975



=== CatBoost ===
accuracy: 0.4981
precision: 0.8039
recall: 0.4708
f1: 0.5938



=== NGBoost ===
accuracy: 0.4975
precision: 0.8038
recall: 0.4698
f1: 0.5930


In [6]:
def rank_metrics_por_usuario(df_scores, top_k=10):
    """
    df_scores: DataFrame con columnas ['customer_id', 'score', 'label'].
    Calcula MAP@K y NDCG@K rankeando los candidatos DE CADA usuario por separado
    y promedia sobre los usuarios que tienen al menos un positivo evaluado.
    """
    aps, ndcgs = [], []

    for _, grupo in df_scores.groupby('customer_id'):
        grupo = grupo.sort_values('score', ascending=False)
        relevantes = grupo['label'].to_numpy()
        n_relevantes = relevantes.sum()
        if n_relevantes == 0:
            continue

        hits = 0
        ap = 0.0
        dcg = 0.0
        for idx, es_relevante in enumerate(relevantes[:top_k]):
            if es_relevante:
                hits += 1
                ap += hits / (idx + 1)
                dcg += 1 / np.log2(idx + 2)
        ap /= min(top_k, n_relevantes)

        ideal_hits = min(top_k, n_relevantes)
        idcg = sum(1 / np.log2(i + 2) for i in range(int(ideal_hits)))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        aps.append(ap)
        ndcgs.append(ndcg)

    return {
        'map@k': float(np.mean(aps)) if aps else 0.0,
        'ndcg@k': float(np.mean(ndcgs)) if ndcgs else 0.0,
        'n_usuarios_evaluados': len(aps),
    }


def obtener_proba_positiva(modelo, X):
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    # NGBoost (clase base, no NGBClassifier) no tiene predict_proba: se obtiene
    # de la distribución predicha.
    return modelo.pred_dist(X).class_probs()[:, 1]


for nombre, modelo in models_to_test:
    proba = obtener_proba_positiva(modelo, X_test)
    df_scores = pd.DataFrame({
        'customer_id': test_df['customer_id'].to_numpy(),
        'score': proba,
        'label': y_test.to_numpy(),
    })
    rank_metrics = rank_metrics_por_usuario(df_scores, top_k=10)

    print(f'\n=== {nombre} (ranking por usuario) ===')
    for k, v in rank_metrics.items():
        print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')



=== LightGBM (ranking por usuario) ===
map@k: 0.8754
ndcg@k: 0.9257
n_usuarios_evaluados: 13872



=== XGBoost (ranking por usuario) ===
map@k: 0.8635
ndcg@k: 0.9187
n_usuarios_evaluados: 13872



=== CatBoost (ranking por usuario) ===
map@k: 0.8633
ndcg@k: 0.9185
n_usuarios_evaluados: 13872



=== NGBoost (ranking por usuario) ===
map@k: 0.8650
ndcg@k: 0.9197
n_usuarios_evaluados: 13872


## 🤝 Collaborative Filtering (SVD / Matrix Factorization)

A diferencia de LightGBM/XGBoost/CatBoost/NGBoost, que clasifican pares usuario-producto a partir de features explícitos, el collaborative filtering aprende directamente de los patrones de interacción en la matriz usuario-item, sin usar features de contenido.

Se aplica `TruncatedSVD` para factorizar la matriz en factores latentes de usuario y de producto, y se reconstruye una matriz de scores predichos como el producto de esos factores.

La matriz se arma únicamente con interacciones del período de train (mismo corte temporal que los clasificadores), así que el SVD nunca ve directamente lo que tiene que predecir. El umbral de decisión se aprende con una regresión logística ajustada solo con pares de train, en vez de elegirse mirando la distribución del test.

In [7]:
from sklearn.decomposition import TruncatedSVD

# Matriz usuario-item construida SOLO con interacciones de train (mismo corte
# temporal que los clasificadores), para que el SVD no vea nada del período de test.
event_weights = {'page_view': 1, 'add_to_cart': 3, 'purchase': 5}
events_train_scored = events_train.copy()
events_train_scored['score'] = events_train_scored['event_type'].map(event_weights).fillna(1)

matriz_train = (
    events_train_scored
    .groupby(['customer_id', 'product_id'])['score']
    .sum()
    .reset_index()
)

interaction_matrix_train = matriz_train.pivot_table(
    index='customer_id', columns='product_id', values='score', fill_value=0
).reindex(index=customer_ids_unique, columns=product_ids_unique, fill_value=0)

print('interaction_matrix_train shape:', interaction_matrix_train.shape)

customer_index = {cid: i for i, cid in enumerate(interaction_matrix_train.index)}
product_index = {pid: j for j, pid in enumerate(interaction_matrix_train.columns)}

svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(interaction_matrix_train.to_numpy())
item_factors = svd.components_
reconstructed_scores = user_factors @ item_factors

print(f'Varianza explicada por los {svd.n_components} componentes: {svd.explained_variance_ratio_.sum():.4f}')


def score_cf(customer_id, product_id):
    i, j = customer_index.get(customer_id), product_index.get(product_id)
    if i is None or j is None:
        return 0.0
    return reconstructed_scores[i, j]


train_pairs = train_pairs.copy()
test_pairs = test_pairs.copy()
train_pairs['cf_score'] = [score_cf(c, p) for c, p in zip(train_pairs['customer_id'], train_pairs['product_id'])]
test_pairs['cf_score'] = [score_cf(c, p) for c, p in zip(test_pairs['customer_id'], test_pairs['product_id'])]

# Umbral de decisión: se aprende con una regresión logística de 1 variable sobre
# los pares de TRAIN, igual que cualquier otro clasificador de esta comparación
# (antes se elegía mirando la cantidad de positivos del propio test, lo cual
# forzaba precision == recall == f1 por construcción).
calibrador = LogisticRegression(class_weight='balanced')
calibrador.fit(train_pairs[['cf_score']], train_pairs['label'])
cf_pred = calibrador.predict(test_pairs[['cf_score']])

metrics_cf = {
    'accuracy': accuracy_score(test_pairs['label'], cf_pred),
    'precision': precision_score(test_pairs['label'], cf_pred, zero_division=0),
    'recall': recall_score(test_pairs['label'], cf_pred, zero_division=0),
    'f1': f1_score(test_pairs['label'], cf_pred, zero_division=0),
}

print('\n=== Collaborative Filtering (SVD) ===')
for k, v in metrics_cf.items():
    print(f'{k}: {v:.4f}')

rank_metrics_cf = rank_metrics_por_usuario(
    test_pairs.rename(columns={'cf_score': 'score'})[['customer_id', 'score', 'label']],
    top_k=10,
)
print('\n=== Collaborative Filtering (ranking por usuario) ===')
for k, v in rank_metrics_cf.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')


interaction_matrix_train shape: (20000, 1197)


Varianza explicada por los 20 componentes: 0.0412



=== Collaborative Filtering (SVD) ===
accuracy: 0.3312
precision: 0.7576
recall: 0.2084
f1: 0.3269



=== Collaborative Filtering (ranking por usuario) ===
map@k: 0.8260
ndcg@k: 0.8914
n_usuarios_evaluados: 13872


## 📊 Comparación de resultados

Los resultados se comparan en dos niveles:
- métricas de clasificación estándar: accuracy, precision, recall y F1
- métricas de ranking: MAP@K y NDCG@K, que son más cercanas al uso real de un recomendador

El modelo con mejor desempeño en estas métricas se convierte en el candidato principal para la siguiente iteración.